# Milestone 1: Data Profiling

**Tegar Reski Pratama — 202310370311215**

Adaptasi template UMM-GURU untuk *Indonesian News Articles Published at 2017*.
Unduh `mq_news.csv` sesuai `data/README.md`, lalu jalankan setiap cell dari atas.
File ini disiapkan tanpa execution count dan tanpa output.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents)
     if (path / 'requirements.txt').is_file() and (path / 'data/README.md').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Buka Jupyter dari akar repository atau folder notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
from IPython.display import Markdown, display
from src.common import (
    connect_analytics, prepare_directories, read_json, sha256_file,
    sink_parquet_atomic, sql_literal, write_json,
)
prepare_directories(PROJECT_ROOT)

from src.dataset import DATASET_URL, missing_summary, scan_news_csv

DATA_CANDIDATES = [
    PROJECT_ROOT / 'data/raw/Dataset Indonesian News Articles Published at 2017/mq_news.csv',
    PROJECT_ROOT / 'data/raw/mq_news.csv',
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), DATA_CANDIDATES[0])
RAW_PARQUET = PROJECT_ROOT / 'data/processed/dataset_raw.parquet'
# Pemetaan otomatis mengenali date/title/content/source dan variasi headernya.
# Dictionary kosong memang berarti memakai pemetaan otomatis.
COLUMN_OVERRIDES = {}

if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Unduh CSV asli dari {DATASET_URL} ke {DATA_PATH}.')

raw_csv, lazy_data, column_mapping = scan_news_csv(DATA_PATH, COLUMN_OVERRIDES)
print('Header CSV:', raw_csv.collect_schema().names())
print('Pemetaan kolom:', column_mapping)
print(f'Ukuran CSV: {DATA_PATH.stat().st_size / 1_000_000:,.2f} MB')


## Profil Polars

`scan_csv` memakai lazy evaluation. Kolom CSV dibaca sebagai String terlebih dahulu
agar inferensi tipe tidak merusak teks atau tanggal; konversi tipe dilakukan saat cleaning.


In [ ]:
raw_schema = raw_csv.collect_schema()
row_count = int(lazy_data.select(pl.len()).collect(engine='streaming').item())
size_bytes = DATA_PATH.stat().st_size
size_requirement_met = size_bytes >= 500_000_000 or row_count > 1_000_000
ringkasan = pl.DataFrame({
    'jumlah_baris': [row_count], 'jumlah_kolom_csv': [len(raw_schema)],
    'ukuran_csv_MB': [size_bytes / 1_000_000],
    'memenuhi_syarat_ukuran': [size_requirement_met],
})
display(ringkasan)
assert row_count > 0, 'CSV tidak memiliki artikel.'
assert size_requirement_met, 'CSV harus >= 500 MB atau memiliki > 1.000.000 baris.'


In [ ]:
null_summary = missing_summary(
    lazy_data, ['published_raw', 'title_raw', 'content_raw', 'source_raw', 'url_raw']
)
blank_summary = lazy_data.select([
    pl.col(name).str.strip_chars().eq('').fill_null(False).sum().alias(name)
    for name in ['published_raw', 'title_raw', 'content_raw', 'source_raw', 'url_raw']
]).collect(engine='streaming')
display(null_summary)
display(blank_summary)
null_summary.write_csv(PROJECT_ROOT / 'output/nulls_raw.csv')


In [ ]:
contoh_data = lazy_data.head(10).with_columns(
    pl.col('content_raw').str.slice(0, 160).alias('content_preview')
).select('row_id', 'published_raw', 'title_raw', 'source_raw', 'content_preview')
contoh_data.collect(engine='streaming')


## Konversi CSV ke Parquet

Simpan salinan kolom analisis beserta `row_id` dengan sink streaming.
CSV asli dipertahankan; checksum mengidentifikasi file input secara persis.


In [ ]:
raw_csv_sha256 = sha256_file(DATA_PATH)
sink_parquet_atomic(lazy_data, RAW_PARQUET)
raw_parquet_sha256 = sha256_file(RAW_PARQUET)
print(f'Parquet tersimpan: {RAW_PARQUET.relative_to(PROJECT_ROOT)}')
print(f'SHA-256 CSV: {raw_csv_sha256}')


## Profil DuckDB

`SUMMARIZE` menghasilkan statistik tipe, null, min/max, dan perkiraan nilai unik.
Nilai unik pada output ini bersifat aproksimasi; deduplikasi memakai kunci artikel secara eksplisit.


In [ ]:
connection = connect_analytics(PROJECT_ROOT)
connection.execute(
    f'CREATE VIEW raw_news AS SELECT * FROM read_parquet({sql_literal(RAW_PARQUET)})'
)
profil_duckdb = connection.execute('SUMMARIZE SELECT * FROM raw_news').pl()
profil_duckdb.write_csv(PROJECT_ROOT / 'output/profile_raw.csv')
display(profil_duckdb)
assert connection.execute('SELECT COUNT(*) FROM raw_news').fetchone()[0] == row_count
connection.close()


## Catatan Milestone 1

Ringkasan berikut dihitung dari CSV lokal dan disimpan untuk pemeriksaan notebook 02.
Kolom tambahan di luar tanggal, judul, isi, sumber, dan URL tidak disalin ke Parquet analisis;
header asli tetap tercatat dalam metadata dan CSV asli.


In [ ]:
profiling_summary = {
    'dataset_url': DATASET_URL,
    'raw_csv': DATA_PATH.relative_to(PROJECT_ROOT).as_posix(), 'raw_size_bytes': size_bytes,
    'rows_raw': row_count, 'csv_columns': dict((k, str(v)) for k, v in raw_schema.items()),
    'column_mapping': column_mapping, 'size_requirement_met': size_requirement_met,
    'raw_csv_sha256': raw_csv_sha256, 'raw_parquet_sha256': raw_parquet_sha256,
    'null_counts': null_summary.to_dicts(),
    'blank_counts': blank_summary.to_dicts()[0],
    'polars_version': pl.__version__,
}
write_json(PROJECT_ROOT / 'output/profiling_summary.json', profiling_summary)
highest_null = null_summary.row(0, named=True)
display(Markdown(
    f"CSV berisi **{row_count:,} artikel**, berukuran **{size_bytes / 1_000_000:,.2f} MB**. "
    f"Syarat skala: **{size_requirement_met}**. Kolom kanonis dengan null terbanyak: "
    f"**{highest_null['kolom']}** ({highest_null['jumlah_null']:,} null). "
    "Kolom opsional yang tidak tersedia akan tampil sebagai null; notebook 02 "
    "memeriksa tanggal, teks kosong, duplikat, dan panjang artikel."
))
